<a href="https://colab.research.google.com/github/lizzietsitsishvili/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2013/%5BLab_13%5D_Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/lizzietsitsishvili/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [17]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        20:14:30   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [18]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:14:30   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [19]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [20]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [21]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price,Price_Residuals,Age_Residuals
0,77.5,38.1,684100.56,-56823.332740,0.621803
1,11.0,95.1,413634.22,19000.990249,-13.689856
2,47.7,73.5,456709.35,-69149.815200,3.233510
3,61.9,60.3,624533.95,18481.157582,5.347789
4,100.8,16.4,870137.54,-2619.815668,4.053610
...,...,...,...,...,...
995,87.7,10.1,932592.35,21560.763160,-14.814575
996,21.2,91.8,412741.12,-1940.516556,-6.511286
997,96.5,14.5,880901.56,-3398.817768,-1.986001
998,20.1,95.1,396659.79,2026.560249,-4.589856


In [22]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [24]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.graph_objects as go

# Step 1: load the data
url = 'https://raw.githubusercontent.com/lizzietsitsishvili/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

# Step 2: run the multivariate OLS model
model = smf.ols(
    'Sale_Price ~ Property_Age + Distance_to_Tech_Hub',
    data=df
).fit()

# optional: print regression results
print(model.summary())

# Step 3: extract coefficients from the fitted statsmodels model
# Intercept = predicted Sale_Price when both X variables are 0
b0 = model.params['Intercept']
b1 = model.params['Property_Age']
b2 = model.params['Distance_to_Tech_Hub']

# Step 4: create a grid of values for the regression plane
# We take evenly spaced values across the observed range of each independent variable
age_range = np.linspace(df['Property_Age'].min(), df['Property_Age'].max(), 30)
distance_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 30)

# meshgrid creates every possible combination of Property_Age and Distance_to_Tech_Hub
# this gives us the x and y coordinates needed to draw the fitted surface plane
X_age, X_distance = np.meshgrid(age_range, distance_range)

# Step 5: compute predicted Sale_Price on that grid using the regression equation
# Sale_Price_hat = b0 + b1*Property_Age + b2*Distance_to_Tech_Hub
Z_price = b0 + b1 * X_age + b2 * X_distance

# Step 6: build the interactive 3D plot
fig = go.Figure()

# scatter plot of actual observed homes
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(size=4, opacity=0.8),
    name='Observed Data'
))

# fitted OLS regression hyperplane
fig.add_trace(go.Surface(
    x=X_age,
    y=X_distance,
    z=Z_price,
    opacity=0.6,
    name='OLS Regression Plane',
    showscale=False
))

# Step 7: format the chart
fig.update_layout(
    title='3D OLS Regression Hyperplane',
    scene=dict(
        xaxis_title='Property Age',
        yaxis_title='Distance to Tech Hub',
        zaxis_title='Sale Price'
    ),
    width=900,
    height=700
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:15:18   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 